# Exercice 2 : Écrire une règle de détection

> **En une phrase :** une règle de détection, c'est un IoC écrit pour une machine. On en écrit trois, on les teste sur de vrais journaux, et on mesure ce qu'elles **ratent** et ce qu'elles **inventent**.

**Le scénario.** Votre SOC reçoit deux flux : les requêtes d'un **site web** et le journal **SSH** de l'exercice 1. Vous devez écrire les règles qui lèveront des alertes.

**Les sources.**
- *Elastic* (exemples publics) : **10 000 requêtes** réelles reçues par un site web personnel (blogue et logiciels libres à télécharger), en mai 2015.
- *Loghub* : le journal SSH de l'exercice 1.

**Le plan.**
1. Tester la règle Suricata du cours (« User-Agent curl »).
2. Chercher les vrais attaquants que cette règle a ratés.
3. Une règle à **seuil** pour la force brute SSH.
4. Régler le seuil : rater des attaques ou noyer l'analyste ?

## Étape 0 : exécutez cette cellule en premier
Le notebook est **autonome** : il fonctionne tel quel sur Colab (ouvert depuis Drive) ou en local. Elle doit afficher **Prêt**.

In [1]:
import re, requests
import pandas as pd

URL_SSH = "https://raw.githubusercontent.com/logpai/loghub/master/OpenSSH/OpenSSH_2k.log"

def charger_ssh():
    """Télécharge le journal SSH et le range dans un tableau : une ligne = un événement."""
    lignes = requests.get(URL_SSH, timeout=30).text.splitlines()
    motif = re.compile(r"^(\w{3}\s+\d+ \d\d:\d\d:\d\d) \S+ sshd\[\d+\]: (.*)$")
    evenements = []
    for ligne in lignes:
        m = motif.match(ligne)
        if m:
            ip = re.search(r"\d+\.\d+\.\d+\.\d+", m.group(2))
            evenements.append({"heure": pd.to_datetime("2016 " + m.group(1), format="%Y %b %d %H:%M:%S"),  # l'année n'est pas écrite dans le journal
                               "message": m.group(2),
                               "ip": ip.group() if ip else None})
    return pd.DataFrame(evenements), lignes

URL_WEB = "https://raw.githubusercontent.com/elastic/examples/master/Common%20Data%20Formats/apache_logs/apache_logs"

def charger_web():
    """Une ligne de journal Apache → IP, date, méthode, chemin, statut HTTP, User-Agent."""
    motif = re.compile(r'^(\S+) \S+ \S+ \[([^\]]+)\] "(\S+) (\S+)[^"]*" (\d{3}) \S+ "[^"]*" "([^"]*)"')
    lignes = requests.get(URL_WEB, timeout=30).text.splitlines()
    requetes = [m.groups() for m in map(motif.match, lignes) if m]
    web = pd.DataFrame(requetes, columns=["ip", "date", "méthode", "chemin", "statut", "user_agent"])
    web["statut"] = web["statut"].astype(int)
    return web

web = charger_web()
evenements, _ = charger_ssh()
pd.set_option("display.max_colwidth", 70)
print(len(web), "requêtes web,", len(evenements), "événements SSH")
print('Prêt')

9999 requêtes web, 2000 événements SSH
Prêt


## Étape 1 : La règle du cours, sur de vraies données
**Ce qu'on fait :** on reprend la règle Suricata vue en classe :

```
alert http $HOME_NET any -> $EXTERNAL_NET any (msg:"Suspicious User-Agent Detected"; content:"User-Agent|3A| curl"; sid:1000001;)
```

Elle signifie : *« alerte si le User-Agent contient `curl` »*. On l'élargit aux autres outils en ligne de commande (`wget`, `python`), puis on la traduit en Python.

In [2]:
OUTILS = r"curl|wget|python"          # <-- MODIFIABLE : le « content » de la règle

alertes_ua = web[web["user_agent"].str.contains(OUTILS, case=False)]
print(len(alertes_ua), "alertes sur", len(web), "requêtes\n")
alertes_ua[["ip", "chemin", "user_agent"]]

12 alertes sur 9999 requêtes



,ip,chemin,user_agent
3423,178.32.216.134,/projects/keynav/,Python-urllib/2.7
3424,178.32.216.134,/projects/keynav/,Python-urllib/2.7
3574,117.28.234.67,/files/logstash/logstash-1.1.9-monolithic.jar,Wget/1.12 (linux-gnu)
5260,178.32.216.134,/projects/xdotool/,Python-urllib/2.7
5261,178.32.216.134,/projects/xdotool/,Python-urllib/2.7
5339,204.244.74.22,/files/xdotool/xdotool-2.20101014.3063.tar.gz,Wget/1.14 (linux-gnu)
5341,204.244.74.22,/files/xdotool/xdotool-2.20101014.3063.tar.gz,Wget/1.14 (linux-gnu)
5391,82.200.166.110,/files/logstash/logstash-1.1.9-flatjar.jar,Wget/1.13.4 (linux-gnu)
5393,50.7.50.90,/files/xdotool/xdotool-2.20101014.3063.tar.gz,Wget/1.14 (linux-gnu)
5399,50.7.50.90,/files/xdotool/xdotool-2.20101014.3063.tar.gz,Wget/1.14 (linux-gnu)


**👀 Ce qu'on doit voir :** **12 alertes**. En les lisant : des `Wget` et un `curl` qui téléchargent des **logiciels proposés sur le site** (`xdotool-….tar.gz`, `logstash-….jar`), un robot Python qui lit `robots.txt`. Ce sont des utilisateurs normaux : **12 faux positifs sur 12**.

## Étape 2 : Où sont les vrais attaquants ?
**Ce qu'on fait :** ce site n'utilise **pas** WordPress. Quelqu'un qui demande `/wp-login.php` ou `/wp-admin/` cherche donc une page d'administration qui n'existe pas : c'est un **scanner** (Ch. 5, reconnaissance active). On cherche ces chemins.

In [3]:
CHEMINS_SENSIBLES = r"wp-login|wp-admin|/administrator|admin\.php|phpmyadmin|\.\./|/etc/passwd"   # <-- MODIFIABLE

sondes = web[web["chemin"].str.contains(CHEMINS_SENSIBLES, case=False)]
print(len(sondes), "requêtes vers des pages d'administration, depuis", sondes["ip"].nunique(), "IP")
print("Réponses du serveur :", sondes["statut"].value_counts().to_dict())
print("Combien la règle « curl » en a-t-elle vues ?", sondes["user_agent"].str.contains(OUTILS, case=False).sum(), "\n")
sondes["user_agent"].value_counts().head(5)

45 requêtes vers des pages d'administration, depuis 34 IP
Réponses du serveur : {404: 45}
Combien la règle « curl » en a-t-elle vues ? 0 



user_agent
-                                                                                                      27
Mozilla/5.0 (Windows NT 6.0) AppleWebKit/537.1 (KHTML, like Gecko) Chrome/24.0.1309.0 Safari/537.17     6
Mozilla/5.0 (Macintosh; Intel Mac OS X 10.7; rv:22.0) Gecko/20100101 Firefox/22.0                       4
Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.1 (KHTML, like Gecko) Chrome/24.0.1290.1 Safari/537.13     3
Mozilla/5.0 (Windows NT 5.1) AppleWebKit/537.1 (KHTML, like Gecko) Chrome/24.0.1309.0 Safari/537.17     3
Name: count, dtype: int64

**👀 Ce qu'on doit voir :** **45 requêtes** venant de **34 IP**, **toutes en 404** (page introuvable). Et la règle « curl » en a vu **0** : les scanners ne donnent pas de User-Agent du tout (`-`, 27 fois) ou se présentent comme **Chrome ou Firefox**.

**❓ Question pour la classe :** la règle « curl » a 12 alertes, toutes fausses, et rate tous les scanners. Faut-il la supprimer ?

*Réponse : pas forcément, mais il faut savoir ce qu'elle mesure. Le **User-Agent est écrit par le client** : l'attaquant y met ce qu'il veut. Une règle sur un champ que l'attaquant contrôle est **facile à contourner** (une option `-A "Mozilla/5.0"` suffit). La règle « WordPress sur un site sans WordPress » est meilleure car elle repose sur **notre environnement** (ce qui existe chez nous), pas sur la bonne foi du client. C'est le point « Environnement » du cours : il faut connaître son système pour écrire de bons IoC.*

## Étape 3 : Une règle à seuil pour la force brute SSH
**Ce qu'on fait :** on écrit l'IoC du cours *« 5 tentatives de login échouées en 2 minutes »* : pour chaque IP, on compte ses échecs dans une **fenêtre glissante** de 2 minutes.

In [4]:
def regle_force_brute(evenements, seuil=5, fenetre="2min"):
    """Renvoie les IP qui ont fait au moins `seuil` échecs dans une même fenêtre de temps."""
    echecs = evenements[evenements["message"].str.startswith("Failed password")]
    resultat = {}
    for ip, groupe in echecs.groupby("ip"):
        pic = groupe.set_index("heure").sort_index()["message"].rolling(fenetre).count().max()
        resultat[ip] = int(pic)
    pics = pd.Series(resultat, name="pire fenêtre").sort_values(ascending=False)
    return pics[pics >= seuil], pics

detectees, pics = regle_force_brute(evenements, seuil=5)                 # <-- MODIFIABLE
print(f"{len(detectees)} IP déclenchent l'alerte, sur {len(pics)} IP qui ont échoué au moins une fois\n")
detectees

9 IP déclenchent l'alerte, sur 23 IP qui ont échoué au moins une fois



183.62.140.253     61
103.99.0.122       30
112.95.230.3       26
187.141.143.180    23
5.188.10.180       18
185.190.58.151     10
123.235.32.19       7
119.4.203.64        6
60.2.12.12          5
Name: pire fenêtre, dtype: int64

**👀 Ce qu'on doit voir :** **9 IP** sur 23 déclenchent l'alerte, dont les trois grosses de l'exercice 1. La pire, 183.62.140.253, fait **61 échecs en 2 minutes**.

## Étape 4 : Régler le seuil
**Ce qu'on fait :** on rejoue la règle avec plusieurs seuils, puis on regarde les IP qui **échappent** à la règle.

In [5]:
for seuil in [2, 3, 5, 10, 20, 50]:
    print(f"seuil {seuil:>2} échecs / 2 min → {len(pics[pics >= seuil]):>2} IP en alerte")

echecs = evenements[evenements["message"].str.startswith("Failed password")]
duree = echecs.groupby("ip")["heure"].agg(essais="count", premier="min", dernier="max")
duree["étalé sur"] = duree["dernier"] - duree["premier"]
print("\nIP jamais détectées avec le seuil 5, mais insistantes :")
duree[(pics < 5) & (duree["essais"] >= 4)].drop(columns=["premier", "dernier"])

seuil  2 échecs / 2 min → 13 IP en alerte
seuil  3 échecs / 2 min → 11 IP en alerte
seuil  5 échecs / 2 min →  9 IP en alerte
seuil 10 échecs / 2 min →  6 IP en alerte
seuil 20 échecs / 2 min →  4 IP en alerte
seuil 50 échecs / 2 min →  1 IP en alerte

IP jamais détectées avec le seuil 5, mais insistantes :


,essais,étalé sur
ip,,
52.80.34.196,5,0 days 03:13:24


**👀 Ce qu'on doit voir :** de **13 IP** en alerte (seuil 2) à **1 seule** (seuil 50). Et des IP qui passent **sous le radar** : par exemple **52.80.34.196**, 5 essais **étalés sur plus de 3 heures**. Jamais plus de 2 en 2 minutes : la règle ne la verra jamais.

**❓ Question pour la classe :** quel seuil choisiriez-vous ?

*Réponse : il n'y a pas de bonne réponse, seulement un compromis. Seuil bas : on attrape tout, mais un employé qui s'est trompé 2 fois de mot de passe déclenche une alerte (**faux positifs**, analystes noyés). Seuil haut : peu d'alertes, mais l'attaquant patient (« low and slow ») passe (**faux négatifs**). Les attaquants connaissent ces seuils et restent juste en dessous. C'est la limite des règles fixes, et la raison des méthodes par **anomalie** (ex. 4).*

## Pour aller plus loin : la même règle, au format de l'industrie
Les SOC écrivent leurs règles dans un format partagé, **Sigma**, que chaque SIEM (Splunk, Wazuh, Sentinel…) traduit dans son propre langage. Notre règle de l'étape 3 ressemble à ceci :

```yaml
title: Force brute SSH (5 échecs en 2 minutes)
logsource: {product: linux, service: sshd}
detection:
  echec: 'Failed password'
  condition: echec | count() by src_ip > 4
timeframe: 2m
level: medium
tags: [attack.credential_access, attack.t1110.001]
```

Le tag `t1110.001` renvoie à la technique **MITRE ATT&CK** « Brute Force : Password Guessing ». On s'en servira à l'exercice 6.

## Ce qu'il faut retenir
- Une règle de détection = un **IoC + une condition** (contient, compte, seuil, fenêtre de temps).
- Toujours la tester sur de **vraies données** : la règle « curl » du cours donne ici **100 % de faux positifs** et rate tous les scanners.
- Une règle sur un champ que **l'attaquant contrôle** (User-Agent, nom de fichier, IP) se contourne facilement. Une règle fondée sur **notre environnement** (« ce chemin n'existe pas chez nous ») est plus robuste.
- Le **seuil** est un compromis : faux positifs d'un côté, attaquants patients de l'autre.

**Exercice suivant :** au lieu d'écrire nos IoC, utiliser ceux que d'autres ont déjà trouvés (renseignement sur les menaces).